# **Question 5.1.1**
**What problems would arise if the way storage blocks are organised in MySequential were changed to a Python list?**<br><br>
If the way storage blocks are organised in MySequential is changed from an `OrderedDict` to a Python list, the code will run correctly. However, it will not be possible to conveniently access the model’s network structure and parameters using `net.state_dict()`, as is the case with `_modules`.

In [1]:
import torch.nn as nn
import torch

class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            self._modules[str(idx)] = module

    def forward(self, X):
        for block in self._modules.values():
            X = block(X)
        return X

class MySequential_list(nn.Module):
    # use list
    def __init__(self, *args):
        super(MySequential_list, self).__init__()
        self.sequential =[]
        for module in args:
            self.sequential.append(module)

    def forward(self, X):
        for module in self.sequential:
            X = module(X)
        return X

X = torch.rand(1, 10)
net = MySequential(nn.Linear(10, 20), nn.ReLU(), nn.Linear(20,10))
net_list = MySequential_list(nn.Linear(10, 20), nn.ReLU(), nn.Linear(20,10))

print(net(X))
print(net_list(X))
print(net, '\n', net.state_dict())
print(net_list, '\n', net_list.state_dict())

tensor([[ 0.2081, -0.0118,  0.2318,  0.0526, -0.1772,  0.0573,  0.3679,  0.1205,
         -0.0397,  0.0506]], grad_fn=<AddmmBackward0>)
tensor([[ 0.3302, -0.3488, -0.1263, -0.1520, -0.3218,  0.0931, -0.2028,  0.4185,
          0.0955, -0.1410]], grad_fn=<AddmmBackward0>)
MySequential(
  (0): Linear(in_features=10, out_features=20, bias=True)
  (1): ReLU()
  (2): Linear(in_features=20, out_features=10, bias=True)
) 
 OrderedDict({'0.weight': tensor([[-1.3318e-01, -5.5410e-02, -2.8922e-02, -2.2437e-01,  7.3006e-02,
         -2.7681e-01,  1.7380e-01, -6.7498e-02,  1.8879e-01, -8.4605e-02],
        [ 1.9249e-01,  8.0681e-02,  6.2605e-02,  2.8085e-01,  3.2212e-02,
         -3.5545e-02, -2.2401e-01, -2.8024e-01, -1.0411e-01,  9.9608e-02],
        [ 1.0800e-01, -1.8560e-01,  2.4825e-01,  2.8262e-01,  1.3872e-01,
          1.8159e-01,  7.5135e-02, -2.7255e-01,  1.4593e-01,  1.2435e-01],
        [-2.4612e-01,  7.1177e-02, -9.1403e-02,  1.9732e-01,  3.1225e-01,
          2.0108e-01, -9.6193e-02,

# **Question 5.1.2**
**Implement a block that takes two blocks as arguments, for example net1 and net2, and returns the concatenated output of the two networks following forward propagation. This is also known as a parallel block.**

In [2]:
import torch.nn as nn
import torch

class Parallel(nn.Module):
    def __init__(self, net1, net2):
        super().__init__()
        self.net1 = net1
        self.net2 = net2

    def forward(self, X):
        X1 = self.net1(X)
        X2 = self.net2(X)
        return torch.cat((X1, X2), dim=1)

X = torch.rand(2, 10)
net = Parallel(nn.Sequential(nn.Linear(10, 20), nn.ReLU()), nn.Sequential(nn.Linear(10, 20), nn.ReLU()))
output = net(X)
output.size()

torch.Size([2, 40])

# **Question 5.1.3**
**Suppose we want to connect multiple instances on the same network. Implement a function that generates multiple instances of the same block and uses them to build a larger network.**

In [3]:
import torch.nn as nn
import torch

def create_network(num_instances, input_size, hidden_size, output_size):
    linear_layer = nn.Sequential(
        nn.Linear(input_size, hidden_size), nn.ReLU(),
        nn.Linear(hidden_size, input_size)
    )

    instances = [linear_layer for _ in range(num_instances)]
    network = nn.Sequential(*instances)
    
    output_layer = nn.Linear(input_size, output_size)
    network.add_module("output_layer", output_layer)
    return network

net = create_network(num_instances=10, input_size=20, hidden_size=20, output_size=10)
net


Sequential(
  (0): Sequential(
    (0): Linear(in_features=20, out_features=20, bias=True)
    (1): ReLU()
    (2): Linear(in_features=20, out_features=20, bias=True)
  )
  (1): Sequential(
    (0): Linear(in_features=20, out_features=20, bias=True)
    (1): ReLU()
    (2): Linear(in_features=20, out_features=20, bias=True)
  )
  (2): Sequential(
    (0): Linear(in_features=20, out_features=20, bias=True)
    (1): ReLU()
    (2): Linear(in_features=20, out_features=20, bias=True)
  )
  (3): Sequential(
    (0): Linear(in_features=20, out_features=20, bias=True)
    (1): ReLU()
    (2): Linear(in_features=20, out_features=20, bias=True)
  )
  (4): Sequential(
    (0): Linear(in_features=20, out_features=20, bias=True)
    (1): ReLU()
    (2): Linear(in_features=20, out_features=20, bias=True)
  )
  (5): Sequential(
    (0): Linear(in_features=20, out_features=20, bias=True)
    (1): ReLU()
    (2): Linear(in_features=20, out_features=20, bias=True)
  )
  (6): Sequential(
    (0): Linear